## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

from datasets import load_from_disk

from nichejepa import tokenizers

In [3]:
config = {}

config["paths"] = {}
config["experiment"] = {}
config["tokenization"] = {}
config["paths"]["data_directory_path"] = '/lustre/scratch126/cellgen/team361/DATASETS'
config["paths"]["tokenizer_directory_path"] = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer'
config["experiment"]["tokenizer"] = 'cell-graph-tokenizer' # 'cell-neighborhood-tokenizer', 'cell-graph-tokenizer'
config["experiment"]["dataset_name"] = 'mmb0-1b_smb1-1b_2p' # 'mmb0-1b_smb1-1b_1p', 'mmb0-1b_smb1-1b_2p'
config["experiment"]["nproc"] = 2

tokenizer = config["experiment"]["tokenizer"]
dataset_name = config["experiment"]["dataset_name"]
nproc = config["experiment"]["nproc"]

In [4]:
data_directory_path = config["paths"]["data_directory_path"]
st_data_silver_folder_path = f"{data_directory_path}/silver"
st_data_gold_folder_path = f"{data_directory_path}/gold"

In [5]:
# Configure tokenizer paths
tokenizer_directory_path = config["paths"]["tokenizer_directory_path"]
tokenizer_folder_path = f"{tokenizer_directory_path}/{tokenizer}/{dataset_name}"
token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary.pkl"
gene_panel_ID_to_gene_panel_dict_file = f"{tokenizer_folder_path}/gene_panel_ID_to_gene_panel_dict.pkl"
file_path_to_gene_panel_ID_dict_file = f"{tokenizer_folder_path}/file_path_to_gene_panel_ID_dict.pkl"
os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

In [11]:
import pickle

token_dictionary_file = f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenizer}/{dataset_name}/token_dictionary.pkl'

with open(token_dictionary_file, 'rb') as f:
    token_dict = pickle.load(f)

print(token_dict)

{'<pad>': 0, '<mask>': 1, '<cls_0>': 2, '<cls_1>': 3, '<cls_2>': 4, '<cls_3>': 5, '<cls_4>': 6, '<cls_5>': 7, '<cls_6>': 8, '<cls_7>': 9, '<cls_8>': 10, '<cls_9>': 11, '<cls_10>': 12, '<cls_11>': 13, '<cls_12>': 14, '<cls_13>': 15, '<cls_14>': 16, '<cls_15>': 17, '<cls_16>': 18, '<cls_17>': 19, '<cls_18>': 20, '<cls_19>': 21, '<cls_20>': 22, '<cls_21>': 23, '<cls_22>': 24, '<cls_23>': 25, '<cls_24>': 26, '<cls_25>': 27, '<cls_26>': 28, '<cls_27>': 29, '<cls_28>': 30, '<cls_29>': 31, '<cls_30>': 32, '<cls_31>': 33, '<cls_32>': 34, '<cls_33>': 35, '<cls_34>': 36, '<cls_35>': 37, '<cls_36>': 38, '<cls_37>': 39, '<cls_38>': 40, '<cls_39>': 41, '<cls_40>': 42, '<cls_41>': 43, '<cls_42>': 44, '<cls_43>': 45, '<cls_44>': 46, '<cls_45>': 47, '<cls_46>': 48, '<cls_47>': 49, '<cls_48>': 50, '<cls_49>': 51, '<cls_50>': 52, '<cls_51>': 53, '<cls_52>': 54, '<cls_53>': 55, '<cls_54>': 56, '<cls_55>': 57, '<cls_56>': 58, '<cls_57>': 59, '<cls_58>': 60, '<cls_59>': 61, '<cls_60>': 62, '<cls_61>': 63, 

## 2. Neighborhood Tokenization

In [ ]:
# Set input and output directories
input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"

model_input_size = 2170 # 862, 2170
n_neighs = 8
include_zero_expr_genes = False
norm_factor = None # "read_depth", None
norm_method = "shifted_log"
if norm_factor is not None:
    output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}{f'_knn_{n_neighs}' if n_neighs is not None else ''}"
else:
    output_file_prefix = f"{dataset_name}_{norm_method}_{model_input_size}{f'_knn_{n_neighs}' if n_neighs is not None else ''}"

In [ ]:
# Instantiate tokenizer
tk = tokenizers.CellNeighborhoodTokenizer(
        model_input_size=model_input_size,
        include_zero_expr_genes=include_zero_expr_genes,
        n_neighs=n_neighs,
        delaunay=(True if n_neighs is None else False),
        nproc=nproc,
        token_dictionary_file=token_dictionary_file,
        gene_panel_ID_to_gene_panel_dict_file=gene_panel_ID_to_gene_panel_dict_file,
        file_path_to_gene_panel_ID_dict_file=file_path_to_gene_panel_ID_dict_file,
        norm_factor=norm_factor,
        norm_method=norm_method)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                 output_directory=output_directory,
                 output_file_prefix=output_file_prefix)

In [ ]:
if norm_factor is not None:
    dataset = load_from_disk(f'/lustre/scratch126/cellgen/team361/DATASETS/gold/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset')
else:
    dataset = load_from_disk(f'/lustre/scratch126/cellgen/team361/DATASETS/gold/{tokenizer}/{dataset_name}/{dataset_name}_{norm_method}_{model_input_size}.dataset')

In [ ]:
dataset

In [ ]:
len(dataset["seg_tokens"][0])

In [ ]:
len(dataset["gene_tokens"][0])

In [ ]:
len(dataset["gene_expr"][0])

In [ ]:
dataset["seg_tokens"][0]

In [ ]:
dataset["gene_tokens"][0]

In [ ]:
dataset['gene_expr'][0]

In [ ]:
dataset['cls_tokens'][0]

In [ ]:
print(dataset["batch_value"][0])
print(dataset["batch_value"][-1])

## 3. Graph Tokenization

In [6]:
config["experiment"]["tokenizer"] = 'cell-graph-tokenizer'
tokenizer = config["experiment"]["tokenizer"]

In [7]:
# Configure tokenizer paths
tokenizer_directory_path = config["paths"]["tokenizer_directory_path"]
tokenizer_folder_path = f"{tokenizer_directory_path}/{tokenizer}/{dataset_name}"
token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary.pkl"
gene_panel_ID_to_gene_panel_dict_file = f"{tokenizer_folder_path}/gene_panel_ID_to_gene_panel_dict.pkl"
file_path_to_gene_panel_ID_dict_file = f"{tokenizer_folder_path}/file_path_to_gene_panel_ID_dict.pkl"
os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

In [8]:
# Set input and output directories
input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"
model_input_size = 2113 # 1793, 2113
n_neighs = 10
include_zero_expr_genes = False
norm_factor = None # "read_depth", None
norm_method = "shifted_log"
if norm_factor is not None:
    output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}{f'_knn_{n_neighs}' if n_neighs is not None else ''}"
else:
    output_file_prefix = f"{dataset_name}_None_{norm_method}_{model_input_size}{f'_knn_{n_neighs}' if n_neighs is not None else ''}"

In [9]:
# Instantiate tokenizer
tk = tokenizers.CellGraphTokenizer(
        model_input_size=model_input_size,
        include_zero_expr_genes=include_zero_expr_genes,
        n_neighs=n_neighs,
        delaunay=(True if n_neighs is None else False),
        nproc=nproc,
        token_dictionary_file=token_dictionary_file,
        gene_panel_ID_to_gene_panel_dict_file=gene_panel_ID_to_gene_panel_dict_file,
        file_path_to_gene_panel_ID_dict_file=file_path_to_gene_panel_ID_dict_file,
        norm_factor=norm_factor,
        norm_method=norm_method)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                 output_directory=output_directory,
                 output_file_prefix=output_file_prefix)

Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph.
Normalizing gene expression counts.
Retrieving gene tokens.
Ranking gene tokens based on normalized counts.


/lustre/scratch126/cellgen/team361/sb75/nichejepa/src/nichejepa/tokenizers/cell_tokenizers.py:509: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var['ensembl_id'][coding_miRNA_idx]


Retrieving tokens for neighborhood cells.


/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/scipy/sparse/_index.py:108: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph.
Normalizing gene expression counts.
Retrieving gene tokens.
Ranking gene tokens based on normalized counts.


/lustre/scratch126/cellgen/team361/sb75/nichejepa/src/nichejepa/tokenizers/cell_tokenizers.py:509: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var['ensembl_id'][coding_miRNA_idx]


Retrieving tokens for neighborhood cells.


Map (num_proc=2):   0%|          | 0/92520 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/92520 [00:00<?, ? examples/s]

In [ ]:
if norm_factor is not None:
    dataset = load_from_disk(f'/lustre/scratch126/cellgen/team361/DATASETS/gold/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset')
else:
    dataset = load_from_disk(f'/lustre/scratch126/cellgen/team361/DATASETS/gold/{tokenizer}/{dataset_name}/{dataset_name}_None_{norm_method}_{model_input_size}.dataset')

In [ ]:
dataset = load_from_disk(f'/lustre/scratch126/cellgen/team361/DATASETS/gold/{tokenizer}/{dataset_name}/{dataset_name}_8_read_depth_{norm_method}_{model_input_size}.dataset')

In [ ]:
dataset

In [ ]:
seg_tokens = dataset["seg_tokens"][0]
gene_tokens = dataset["gene_tokens"][0]
gene_expr = dataset["gene_expr"][0]

In [ ]:
len(seg_tokens)

In [ ]:
len(gene_tokens)

In [ ]:
len(dataset["gene_expr"][0])

In [ ]:
seg_tokens

In [ ]:
gene_tokens

In [ ]:
counts = []
for gt in dataset['gene_tokens']:
    counts.append(sum([token != 0 for token in gt]))
print(min(counts))

In [ ]:
min_index = counts.index(min(counts))
print(min_index)

In [ ]:
dataset['gene_tokens'][38283]

In [ ]:
dataset['gene_expr'][0]

In [ ]:
dataset['cls_tokens'][0]

In [ ]:
max([len(seq) for seq in dataset['cls_tokens']])

In [ ]:
print(dataset["batch_value"][0])
print(dataset["batch_value"][-1])

## 4. Miscellaneous

In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import pickle

In [ ]:
file_path = "harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad" # "harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad", "harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad"
adata = sc.read_h5ad(f"/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_1p/{file_path}")

sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key="spatial",
                        #radius=27.5
                        delaunay=True
                       )

print(np.mean(adata.obsp["spatial_connectivities"].sum(1)))
print(np.min(adata.obsp["spatial_connectivities"].sum(1)))

In [ ]:
starmap_file_name = "harmonised_starmap_plus_mouse_cns_batch15.h5ad"
adata_starmap = sc.read_h5ad(f"/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_2p/{starmap_file_name}")
merfish_file_name = "harmonised_merfish_mouse_brain_239-batch_batch82.h5ad"
adata_merfish = sc.read_h5ad(f"/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_2p/{merfish_file_name}")

In [ ]:
starmap_genes = adata_starmap.var["ensembl_id"].tolist()
merfish_genes = adata_merfish.var["ensembl_id"].tolist()
print(len(set(starmap_genes + merfish_genes)))
print(len(set(starmap_genes) ^ set(merfish_genes)))

In [ ]:
nonshared_genes = (list(set(starmap_genes) ^ set(merfish_genes)))

In [ ]:
token_dictionary_file = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-neighborhood-tokenizer/mmb0-1b_smb1-1b_2p/token_dictionary.pkl'

with open(token_dictionary_file, 'rb') as f:
    token_dict = pickle.load(f)

In [ ]:
nonshared_tokens = []
for gene in nonshared_genes:
    nonshared_tokens.append(token_dict[gene])

In [ ]:
from pprint import pprint
pprint(nonshared_tokens)

In [ ]:
max(np.count_nonzero(adata_merfish.X.toarray(), axis=1))

In [ ]:
sq.gr.spatial_neighbors(adata_merfish,
                        coord_type="generic",
                        spatial_key="spatial",
                        #radius=27.5
                        delaunay=True
                       )

adata_merfish.layers['X_neighborhood'] = (
        adata_merfish.obsp['spatial_connectivities'].T @ adata_merfish.X)

max(np.count_nonzero(adata_merfish.layers['X_neighborhood'].toarray(), axis=1))

In [ ]:
adata_annot = sc.read_h5ad("/lustre/scratch126/cellgen/team361/ab70/share/anndata_umap_with_clusters.h5ad")

In [ ]:
merged_obs = pd.merge(
    adata_starmap.obs,
    adata_annot.obs[['nichecompass_latent_cluster']],
    left_index=True,
    right_index=True,
    how='left',
)

adata_starmap.obs = merged_obs
adata_starmap.write(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_2p_labels/{starmap_file_name}')

In [ ]:
# Perform a left join on the obs DataFrames based on the index (obs_names)
merged_obs = pd.merge(
    adata_merfish.obs,
    adata_annot.obs[['nichecompass_latent_cluster']],
    left_index=True,
    right_index=True,
    how='left',
)

adata_merfish.obs = merged_obs
adata_merfish.write(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_2p_labels/{merfish_file_name}')

In [ ]:
len(adata_merfish.var)

In [ ]:
len(adata_starmap.var)

In [ ]:
import pickle

token_dictionary_file = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-neighborhood-tokenizer/mmb0-1b_smb1-1b_2p/token_dictionary.pkl'

with open(token_dictionary_file, 'rb') as f:
    token_dict = pickle.load(f)


In [ ]:
token_dict["ENSMUSG00000038331"]